# 🔬 The Formula Behind the Data | Perfect Score on Original Dataset

🧠 The target is not random — it follows a simple rule

Someone brilliant in the [discussion](https://www.kaggle.com/competitions/playground-series-s6e4/discussion/687460) discovered the **exact formula** that generates the original dataset's labels. 

This notebook:
* implements the formula
* verifies it (perfect score on original data)
* uses it as a feature for modeling

**TL;DR**: `Irrigation_Need` is driven by just 6 variables and a simple scoring rule. The competition data follows the same pattern (with noise).

Credit to [@cdeotte](https://www.kaggle.com/cdeotte) for discovering this.

If this helped, **please consider upvoting** 

---

## Why this matters

The dataset is not purely random — it follows underlying logic

By extracting that logic:

* you get stronger features
* better generalization
* more interpretable models

👉 This is signal, not noise

## Imports

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import cross_val_score
import xgboost as xgb

## Load Data

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv', index_col='id')
test  = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv', index_col='id')
orig  = pd.read_csv('/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv')

LABEL_MAP = {'Low': 0, 'Medium': 1, 'High': 2}
train['target'] = train['Irrigation_Need'].map(LABEL_MAP)
orig['target']  = orig['Irrigation_Need'].map(LABEL_MAP)

print(f"Train: {train.shape}, Orig: {orig.shape}, Test: {test.shape}")

## The Formula

The original dataset's target is determined by a simple rule:

**High score** (reasons the field needs MORE water):
- +2 if `Soil_Moisture < 25` (dry soil)
- +2 if `Rainfall_mm < 300` (no rain)
- +1 if `Temperature_C > 30` (hot weather evaporates water)
- +1 if `Wind_Speed_kmh > 10` (wind dries the soil)

**Low score** (reasons the field needs LESS water):
- +2 if `Crop_Growth_Stage` is Harvest (plant is done growing)
- +2 if `Crop_Growth_Stage` is Sowing (seed just planted, low demand)
- +1 if `Mulching_Used = Yes` (mulch retains moisture)

**Decision rule:**
- `Score = High score - Low score`
- Score ≤ 0 → **Low**
- 0 < Score ≤ 3 → **Medium**
- Score > 3 → **High**

This makes complete agronomic sense! Let's verify it.

In [ ]:
def irrigation_formula(df):
    # High score — reasons the field needs MORE water
    high_score = (
        (df['Soil_Moisture'] < 25).astype(int) * 2 +
        (df['Rainfall_mm'] < 300).astype(int) * 2 +
        (df['Temperature_C'] > 30).astype(int) * 1 +
        (df['Wind_Speed_kmh'] > 10).astype(int) * 1
    )
    
    # Low score — reasons the field needs LESS water
    low_score = (
        (df['Crop_Growth_Stage'] == 'Harvest').astype(int) * 2 +
        (df['Crop_Growth_Stage'] == 'Sowing').astype(int) * 2 +
        (df['Mulching_Used'] == 'Yes').astype(int) * 1
    )
    
    # Net score
    score = high_score - low_score
    
    # Decision rule
    prediction = np.where(score <= 0, 0,          # Low
                 np.where(score <= 3, 1, 2))       # Medium / High
    
    return score, prediction

In [ ]:
orig_score, orig_pred = irrigation_formula(orig)

ba = balanced_accuracy_score(orig['target'], orig_pred)
print(f"Balanced Accuracy on original dataset: {ba:.4f}")
print(f"\nPerfect? {'YES' if ba == 1.0 else 'NO'}")
print(f"\nClassification Report:")
print(classification_report(orig['target'], orig_pred, target_names=['Low', 'Medium', 'High']))

## How well does the formula work on the competition's synthetic data?

The competition data was generated by a deep learning model trained on the original. 
So it follows a *similar* pattern, but with noise. Let's check.

In [ ]:
train_score, train_pred = irrigation_formula(train)

ba_train = balanced_accuracy_score(train['target'], train_pred)
print(f"Balanced Accuracy on competition train: {ba_train:.4f}")
print(f"\nClassification Report:")
print(classification_report(train['target'], train_pred, target_names=['Low', 'Medium', 'High']))

## Using the formula as a FEATURE

The formula alone won't win because the synthetic data has noise. But the **formula score** 
is an incredibly powerful feature — it captures the exact generating logic.

We can add it (and its components) as features to any model.

In [ ]:
def add_formula_features(df):
    """Add the formula components as features."""
    df = df.copy()
    
    # Individual binary indicators (the building blocks)
    df['f_dry_soil']    = (df['Soil_Moisture'] < 25).astype(np.int8)
    df['f_low_rain']    = (df['Rainfall_mm'] < 300).astype(np.int8)
    df['f_hot']         = (df['Temperature_C'] > 30).astype(np.int8)
    df['f_windy']       = (df['Wind_Speed_kmh'] > 10).astype(np.int8)
    df['f_harvest']     = (df['Crop_Growth_Stage'] == 'Harvest').astype(np.int8)
    df['f_sowing']      = (df['Crop_Growth_Stage'] == 'Sowing').astype(np.int8)
    df['f_mulched']     = (df['Mulching_Used'] == 'Yes').astype(np.int8)
    
    # Composite scores
    df['f_high_score']  = df['f_dry_soil'] * 2 + df['f_low_rain'] * 2 + df['f_hot'] + df['f_windy']
    df['f_low_score']   = df['f_harvest'] * 2 + df['f_sowing'] * 2 + df['f_mulched']
    df['f_net_score']   = df['f_high_score'] - df['f_low_score']
    
    # The formula's raw prediction (as a feature, not as final answer)
    df['f_formula_pred'] = np.where(df['f_net_score'] <= 0, 0,
                           np.where(df['f_net_score'] <= 3, 1, 2))
    
    return df

train_fe = add_formula_features(train)
print("Formula features added:")
print(train_fe[['f_dry_soil','f_low_rain','f_hot','f_windy','f_harvest','f_sowing',
                'f_mulched','f_high_score','f_low_score','f_net_score','f_formula_pred']].describe().T)

## Model Comparison

In [ ]:
cats = ['Soil_Type','Crop_Type','Crop_Growth_Stage','Season','Irrigation_Type','Water_Source','Mulching_Used','Region']

# WITHOUT formula features
X_base = train.drop(['Irrigation_Need','target'], axis=1).copy()
for c in cats:
    X_base[c] = X_base[c].astype('category')

# WITH formula features
X_formula = add_formula_features(train).drop(['Irrigation_Need','target'], axis=1).copy()
for c in cats:
    X_formula[c] = X_formula[c].astype('category')

y = train['target'].values

model = xgb.XGBClassifier(
    max_depth=6, learning_rate=0.05, n_estimators=500,
    objective='multi:softprob', enable_categorical=True,
    tree_method='hist', device='cuda', random_state=42, verbosity=0,
)

for name, X in [("Without formula features", X_base), ("With formula features", X_formula)]:
    scores = cross_val_score(model, X, y, cv=5, scoring='balanced_accuracy', n_jobs=1)
    print(f"{name}: {np.mean(scores):.5f} ± {np.std(scores):.5f}")

## Graph

In [ ]:
# Show how the net score distributes across actual classes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original dataset — perfect separation
for cls, name, color in [(0,'Low','forestgreen'), (1,'Medium','steelblue'), (2,'High','coral')]:
    mask = orig['target'] == cls
    axes[0].hist(orig_score[mask], bins=range(-5, 8), alpha=0.6, label=name, color=color, edgecolor='white')
axes[0].set_title('Original Dataset — Perfect Separation')
axes[0].set_xlabel('Formula Net Score')
axes[0].set_ylabel('Count')
axes[0].legend()

# Competition dataset — noisy but signal is clear
for cls, name, color in [(0,'Low','forestgreen'), (1,'Medium','steelblue'), (2,'High','coral')]:
    mask = train['target'] == cls
    axes[1].hist(train_score[mask], bins=range(-5, 8), alpha=0.6, label=name, color=color, edgecolor='white')
axes[1].set_title('Competition Dataset — Noisy but Signal is Clear')
axes[1].set_xlabel('Formula Net Score')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.suptitle('Formula Net Score Distribution by Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Key Takeaways

1. The original dataset was generated by a **simple rule-based formula** using just 6 variables
2. The formula achieves **perfect balanced accuracy (1.0)** on the original data
3. On the synthetic competition data, the formula alone scores ~0.93-0.95 — good but not enough due to noise
4. Adding the formula components as **features** to a tree model gives a meaningful boost
5. The most important variables according to the formula: **Soil_Moisture, Rainfall_mm, Crop_Growth_Stage, Mulching_Used** — which matches what we see in feature importance plots!

This explains why `Mulching_Used` and `Crop_Growth_Stage` dominate the feature importance in every XGBoost model — they're literally part of the generating function.

**Practical advice:** Add `f_net_score` and the individual binary flags as features to your model. They encode the exact signal that the original data was built on.

Credit to [@cdeotte](https://www.kaggle.com/cdeotte) for the formula discovery.

If this was useful, **please consider upvoting** 🙌

Related:
- Baseline model: https://www.kaggle.com/code/simarbirsinghsandhu/xgboost-optuna-gpu-cv-0-972
- Threshold tuning (free score boost🤫): https://www.kaggle.com/code/simarbirsinghsandhu/threshold-tuning-free-score-boost-no-retrain